# RazorShield Sentinel — Stratified Evaluation & Ablation Study

## Honest PR-AUC & Generalization Metrics (Technical Review Fix 1 & Fix 2)

This notebook computes:
1. **Full-Funnel Fraud Catch Rate** (Canary Honeytokens + Deterministic Rules + ML Pipeline)
2. **ML-Layer PR-AUC** (evaluated strictly on ambiguous transactions reaching ML)
3. **Adversarial-Realistic PR-AUC** (stealth carding bots mimicking human biometrics)
4. **Leave-One-Attack-Type-Out Generalization Recall** (trained without CVV-cycling examples)
5. **Ensemble Component Ablation Study** (empirical justification for 0.70/0.20/0.10 weighting)
6. **False Positive Cost & Net_Value_Protected** (§4.2 pitch metric)

In [ ]:
import sys, math, datetime
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
    average_precision_score, roc_curve, precision_recall_curve
)
import lightgbm as lgb
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

In [ ]:
# ── Load dataset and re-engineer features ───────────────────────────────────
from backend.models.train import _engineer_features, FEATURE_COLS

df = pd.read_csv('../data/synthetic_transactions.csv')
df = _engineer_features(df)

X = df[FEATURE_COLS].values.astype(np.float32)
y = df['is_fraud'].values.astype(int)

train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.20, stratify=y, random_state=42
)

X_test, y_test = X[test_idx], y[test_idx]
df_test = df.iloc[test_idx].copy()
seg_attack_types = df['attack_type'].iloc[test_idx].values

print(f'Test set: {len(X_test)} rows | Fraud rate: {y_test.mean():.2%}')

In [ ]:
# ── Load trained models and generate predictions ───────────────────────────
with open('../backend/models/lgbm_model.pkl', 'rb') as f:
    lgbm = pickle.load(f)
with open('../backend/models/if_model.pkl', 'rb') as f:
    if_data = pickle.load(f)
    iso = if_data['model']
    if_min = if_data['score_min']
    if_range = if_data['score_range']

lgbm_probs  = lgbm.predict_proba(X_test)[:, 1]
raw_if      = iso.score_samples(X_test)
if_norm     = np.clip(1.0 - (raw_if - if_min) / max(if_range, 1e-6), 0, 1)
cluster_col = FEATURE_COLS.index('cluster_risk_score')
final_risk  = np.clip(0.70 * lgbm_probs + 0.20 * if_norm + 0.10 * X_test[:, cluster_col], 0, 1)
combined_preds = (final_risk >= 0.50).astype(int)

print('Models loaded and test set scored.')

In [ ]:
# ── 1. Stratified & Honest PR-AUC Breakdown ────────────────────────────────
print('=== 1. Stratified Metric Breakdown ===')

# Naive overall PR-AUC
naive_pr_auc = average_precision_score(y_test, final_risk)
print(f'Naive Overall Test PR-AUC        : {naive_pr_auc:.4f}')

# ML-Layer PR-AUC (excluding deterministic rule overrides)
deterministic_rule_mask = (
    df_test['asn_type'].isin(['datacenter', 'tor'])
    & (df_test['keystroke_entropy'] < 0.1)
    & (df_test['mouse_jitter_score'] < 0.05)
    & (df_test['time_on_page_s'] < 1.0)
    & (df_test['ja3_ua_mismatch'] == 1.0)
).values

ml_only_mask = ~deterministic_rule_mask
ml_pr_auc = average_precision_score(y_test[ml_only_mask], final_risk[ml_only_mask])
print(f'ML-Layer PR-AUC (Ambiguous Only) : {ml_pr_auc:.4f}  (n={ml_only_mask.sum()}/{len(y_test)})')

# Adversarial-realistic PR-AUC (Stealth bots with human-mimicking biometrics)
adv_mask = (seg_attack_types == 'adversarial_realistic') | (y_test == 0)
adv_pr_auc = average_precision_score(y_test[adv_mask], final_risk[adv_mask])
adv_rec = recall_score(y_test[seg_attack_types == 'adversarial_realistic'], combined_preds[seg_attack_types == 'adversarial_realistic'])
print(f'Adversarial-Realistic PR-AUC     : {adv_pr_auc:.4f}  (Recall: {adv_rec:.2%})')

# Full funnel catch rate
funnel_preds = combined_preds.copy()
funnel_preds[deterministic_rule_mask] = 1
print(f'Full-Funnel Fraud Catch Rate     : {recall_score(y_test, funnel_preds):.2%}')

In [ ]:
# ── 2. Leave-One-Attack-Type-Out Generalization Evaluation ─────────────────
print('=== 2. Generalization Evaluation (Trained without CVV-cycling) ===')

train_gen_mask = df['attack_type'] != 'cvv_cycling'
df_train_gen = df[train_gen_mask]
df_test_unseen = df[df['attack_type'] == 'cvv_cycling']

X_train_g = df_train_gen[FEATURE_COLS].values.astype(np.float32)
y_train_g = df_train_gen['is_fraud'].values.astype(int)
X_test_unseen = df_test_unseen[FEATURE_COLS].values.astype(np.float32)
y_test_unseen = df_test_unseen['is_fraud'].values.astype(int)

lgb_gen = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, verbose=-1, random_state=42).fit(X_train_g, y_train_g)
iso_gen = IsolationForest(n_estimators=150, contamination=0.2, random_state=42).fit(X_train_g)

raw_if_unseen = iso_gen.score_samples(X_test_unseen)
if_norm_unseen = np.clip(1.0 - (raw_if_unseen - if_min) / max(if_range, 1e-6), 0, 1)
gen_risk = np.clip(0.70 * lgb_gen.predict_proba(X_test_unseen)[:, 1] + 0.20 * if_norm_unseen + 0.10 * X_test_unseen[:, cluster_col], 0, 1)
gen_preds = (gen_risk >= 0.50).astype(int)

unseen_recall = recall_score(y_test_unseen, gen_preds, zero_division=0)
print(f'Generalization Recall on Unseen Attack Type : {unseen_recall:.2%} (n={len(df_test_unseen)})')
print(f'Average Anomaly Score on Unseen Pattern     : {gen_risk.mean():.4f}')

In [ ]:
# ── 3. Ensemble Weight Ablation Study ──────────────────────────────────────
print('=== 3. Ensemble Weight Ablation Study ===\n')

ablation_results = []
configs = [
    ('Full Ensemble (0.70 LGB / 0.20 IF / 0.10 Clust)', 0.70, 0.20, 0.10),
    ('No IsolationForest (0.85 LGB / 0.00 IF / 0.15 Clust)', 0.85, 0.00, 0.15),
    ('No Cluster Score   (0.75 LGB / 0.25 IF / 0.00 Clust)', 0.75, 0.25, 0.00),
    ('No LightGBM (IF + Cluster Only: 0.00 / 0.65 / 0.35)', 0.00, 0.65, 0.35),
    ('Single LightGBM (1.00 LGB / 0.00 / 0.00)', 1.00, 0.00, 0.00),
]

for name, w_lgb, w_if, w_cl in configs:
    score_abl = np.clip(w_lgb * lgbm_probs + w_if * if_norm + w_cl * X_test[:, cluster_col], 0, 1)
    preds_abl = (score_abl >= 0.50).astype(int)
    pr_abl = average_precision_score(y_test, score_abl)
    rec_abl = recall_score(y_test, preds_abl)
    f1_abl = f1_score(y_test, preds_abl)
    ablation_results.append({
        'Configuration': name,
        'PR-AUC': round(pr_abl, 4),
        'Recall': f'{rec_abl:.2%}',
        'F1': round(f1_abl, 4)
    })

df_abl = pd.DataFrame(ablation_results)
print(df_abl.to_string(index=False))

In [ ]:
# ── 4. Net_Value_Protected Revenue Impact ──────────────────────────────────
CONVERSION_PROB = 0.70
RECOVERY_RATE   = 0.55

genuine_mask = y_test == 0
fp_mask      = (combined_preds == 1) & genuine_mask
fp_amounts   = df_test.loc[genuine_mask, 'amount'].values[combined_preds[genuine_mask] == 1]

fraud_mask   = y_test == 1
tp_mask      = (combined_preds == 1) & fraud_mask
tp_amounts   = df_test.loc[fraud_mask, 'amount'].values[combined_preds[fraud_mask] == 1]

soft_risk_fp  = fp_amounts[final_risk[fp_mask] < 0.50]
fraud_loss_prevented = float(tp_amounts.sum())
fpc_before_recovery  = float(sum(a * CONVERSION_PROB for a in fp_amounts))
recovered_gmv        = float(sum(a * CONVERSION_PROB * RECOVERY_RATE for a in soft_risk_fp))
net_value_protected  = fraud_loss_prevented - (fpc_before_recovery - recovered_gmv)

print('=== 4. Revenue Impact Metrics ===')
print(f'Fraud Loss Prevented       : Rs.{fraud_loss_prevented:>12,.2f}')
print(f'False Positive Cost (FPC)  : Rs.{fpc_before_recovery:>12,.2f}')
print(f'Recovered GMV (Soft-Risk)    : Rs.{recovered_gmv:>12,.2f}')
print(f'NET VALUE PROTECTED        : Rs.{net_value_protected:>12,.2f}')

In [ ]:
# ── 5. Plot Stratified Precision-Recall Curves ─────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

prec_all, rec_all, _ = precision_recall_curve(y_test, final_risk)
prec_ml, rec_ml, _ = precision_recall_curve(y_test[ml_only_mask], final_risk[ml_only_mask])
prec_adv, rec_adv, _ = precision_recall_curve(y_test[adv_mask], final_risk[adv_mask])

ax1.plot(rec_all, prec_all, label=f'Full Pipeline (PR-AUC={naive_pr_auc:.3f})', color='#10b981', lw=2)
ax1.plot(rec_ml, prec_ml, label=f'ML-Layer Only (PR-AUC={ml_pr_auc:.3f})', color='#6366f1', lw=2, linestyle='--')
ax1.plot(rec_adv, prec_adv, label=f'Adversarial Realistic (PR-AUC={adv_pr_auc:.3f})', color='#f59e0b', lw=2, linestyle=':')
ax1.set_xlabel('Recall')
ax1.set_ylabel('Precision')
ax1.set_title('Stratified Precision-Recall Curves')
ax1.legend()
ax1.grid(True, alpha=0.3)

fpr, tpr, _ = roc_curve(y_test, final_risk)
ax2.plot(fpr, tpr, label=f'Combined ROC-AUC ({naive_roc_auc:.3f})', color='#3b82f6', lw=2)
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/razorshield_curves.png', dpi=150)
plt.show()
print('Curves saved to data/razorshield_curves.png')